In [4]:
import pandas as pd
import numpy as np
import time

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error

from surprise import Dataset, Reader, SVDpp
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise.accuracy import rmse

In [5]:
ratings = pd.read_csv("data/ratings_small.csv")
links = pd.read_csv("data/links_small.csv")
movies_metadata = pd.read_csv("data/movies_metadata.csv", low_memory=False)

In [6]:
ratings = ratings[["userId", "movieId", "rating"]]
links = links[["movieId", "tmdbId"]]
movies_metadata = movies_metadata[["id", "title"]]

In [7]:
links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")
movies_metadata["id"] = pd.to_numeric(movies_metadata["id"], errors="coerce")
links = links.dropna()
movies_metadata = movies_metadata.dropna()

In [8]:
links["tmdbId"] = links["tmdbId"].astype(int)
movies_metadata["id"] = movies_metadata["id"].astype(int)

In [9]:
movie_map = links.merge(movies_metadata, left_on="tmdbId", right_on="id", how="inner")
movie_map = movie_map[["movieId", "title"]].drop_duplicates()

In [10]:
full = ratings.merge(movie_map, on="movieId", how="inner")
full = full.dropna()
full = full.drop_duplicates(subset=["userId", "movieId"])
full.head()

,userId,movieId,rating,title
0,1,31,2.5,Dangerous Minds
1,1,1029,3.0,Dumbo
2,1,1061,3.0,Sleepers
3,1,1129,2.0,Escape from New York
4,1,1172,4.0,Cinema Paradiso


In [11]:
pivot = full.pivot_table(index="userId", columns="movieId", values="rating")
pivot_filled = pivot.fillna(0)
matrix = csr_matrix(pivot_filled.values)
matrix

<671x9025 sparse matrix of type '<class 'numpy.float64'>'
	with 99810 stored elements in Compressed Sparse Row format>

KNN

In [12]:
start_knn = time.time()
knn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=6)
knn.fit(matrix)
end_knn = time.time()
knn_time = end_knn - start_knn

In [13]:
def recommend_knn(user_id, n=10):
    if user_id not in pivot_filled.index:
        return "Такого userId нет"

    user_index = list(pivot_filled.index).index(user_id)
    distances, indices = knn.kneighbors(pivot_filled.iloc[user_index, :].values.reshape(1, -1), n_neighbors=6)

    neighbor_ids = pivot_filled.index[indices.flatten()[1:]]
    
    neighbor_ratings = pivot_filled.loc[neighbor_ids]
    mean_ratings = neighbor_ratings.mean(axis=0)

    user_seen = pivot_filled.loc[user_id]
    unseen_movies = user_seen[user_seen == 0].index

    recommendations = mean_ratings[unseen_movies].sort_values(ascending=False).head(n)

    result = full[["movieId", "title"]].drop_duplicates()
    result = result[result["movieId"].isin(recommendations.index)].copy()
    result["predicted_rating"] = result["movieId"].map(recommendations)
    result = result.sort_values("predicted_rating", ascending=False)

    return result[["title", "predicted_rating"]].reset_index(drop=True)

recommend_knn(1, 10)

,title,predicted_rating
0,Beverly Hills Cop,2.0
1,Junior,1.6
2,Pulp Fiction,1.0
3,The Brady Bunch Movie,1.0
4,The Godfather,1.0
5,The Godfather: Part II,1.0
6,Three Kings,1.0
7,Fargo,1.0
8,Ronin,1.0
9,Ali G Indahouse,1.0


Оценка KNN

In [14]:
def predict_knn(user_id, movie_id):
    user_index = list(pivot_filled.index).index(user_id)
    distances, indices = knn.kneighbors(pivot_filled.iloc[user_index, :].values.reshape(1, -1),n_neighbors=6)
    neighbor_ids = pivot_filled.index[indices.flatten()[1:]]
    neighbor_ratings = pivot_filled.loc[neighbor_ids, movie_id]
    neighbor_ratings = neighbor_ratings[neighbor_ratings > 0]
    if len(neighbor_ratings) == 0:
        return None
    return neighbor_ratings.mean()

true_vals = []
pred_vals = []
for _, row in full.iterrows():
    pred = predict_knn(row["userId"], row["movieId"])
    if pred is not None:
        true_vals.append(row["rating"])
        pred_vals.append(pred)
mae_knn = mean_absolute_error(true_vals, pred_vals)
mae_knn

0.806262257365588

SVD

In [15]:
user_means = pivot.mean(axis=1)
pivot_centered = pivot.sub(user_means, axis=0)
pivot_centered = pivot_centered.fillna(0)

In [16]:
start_svd = time.time()

k = 20
svd = TruncatedSVD(n_components=k, random_state=42)
U = svd.fit_transform(pivot_centered)
Vt = svd.components_
R_hat_centered = U @ Vt
R_hat = pd.DataFrame(R_hat_centered, index=pivot.index, columns=pivot.columns)
R_hat = R_hat.add(user_means, axis=0)

end_svd = time.time()
svd_time = end_svd - start_svd

In [17]:
def recommend_svd(user_id, n=10):
    if user_id not in R_hat.index:
        return "Такого userId нет"

    user_pred = R_hat.loc[user_id]
    real_ratings = pivot.loc[user_id]

    unseen = real_ratings.isna()
    rec = user_pred[unseen]
    top = rec.sort_values(ascending=False).head(n)

    result = full[["movieId", "title"]].drop_duplicates()
    result = result[result["movieId"].isin(top.index)].copy()
    result["predicted_rating"] = result["movieId"].map(top)
    result = result.sort_values("predicted_rating", ascending=False)

    return result[["title", "predicted_rating"]].reset_index(drop=True)

recommend_svd(1, 10)

,title,predicted_rating
0,2001: A Space Odyssey,2.581696
1,Pulp Fiction,2.572746
2,Annie Hall,2.569859
3,Dr. Strangelove or: How I Learned to Stop Worr...,2.569815
4,The Empire Strikes Back,2.568239
5,Terminator 2: Judgment Day,2.568203
6,The Rocky Horror Picture Show,2.567002
7,The Terminator,2.566614
8,The Matrix,2.566019
9,Crumb,2.565975


In [18]:
true_vals = []
pred_vals = []

for user_id in pivot.index:
    for movie_id in pivot.columns:
        real = pivot.loc[user_id, movie_id]
        if not np.isnan(real):
            true_vals.append(real)
            pred_vals.append(R_hat.loc[user_id, movie_id])

mae_svd = mean_absolute_error(true_vals, pred_vals)
mae_svd

0.5413224355340498

SVD++

In [19]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(full[["userId", "movieId", "rating"]], reader)
trainset, testset = surprise_train_test_split(data, test_size=0.2, random_state=42)

In [20]:
start_svdpp = time.time()
svdpp = SVDpp(random_state=42)
svdpp.fit(trainset)
end_svdpp = time.time()
svdpp_time = end_svdpp - start_svdpp

In [21]:
def recommend_svdpp(user_id, n=10):
    all_movies = full["movieId"].unique()
    watched = full[full["userId"] == user_id]["movieId"].unique()

    unseen_movies = [movie for movie in all_movies if movie not in watched]

    predictions = []
    for movie_id in unseen_movies:
        pred = svdpp.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))

    predictions = sorted(predictions, key=lambda x: x[1], reverse=True)[:n]

    result = full[["movieId", "title"]].drop_duplicates()
    result = result[result["movieId"].isin([x[0] for x in predictions])].copy()
    result["predicted_rating"] = result["movieId"].map(dict(predictions))
    result = result.sort_values("predicted_rating", ascending=False)

    return result[["title", "predicted_rating"]].reset_index(drop=True)
    
recommend_svdpp(1, 10)

,title,predicted_rating
0,Das Boot,3.706240
1,Modern Times,3.683696
2,The Shawshank Redemption,3.670602
3,Ran,3.623186
4,A Close Shave,3.569732
5,The African Queen,3.553931
6,"Lock, Stock and Two Smoking Barrels",3.541618
7,Gladiator 1992,3.539638
8,Snatch,3.532487
9,Swingers,3.520218


In [22]:
predictions_svdpp = svdpp.test(testset)
true = [pred.r_ui for pred in predictions_svdpp]
pred = [pred.est for pred in predictions_svdpp]

mae_svdpp = mean_absolute_error(true, pred)
mae_svdpp

0.6821933292833064

In [23]:
comparison = pd.DataFrame({
    "Algorithm": ["KNN", "SVD", "SVD++"],
    "MAE": [mae_knn, mae_svd, mae_svdpp],
    "Time": [knn_time, svd_time, svdpp_time]
})

comparison

,Algorithm,MAE,Time
0,KNN,0.806262,0.001001
1,SVD,0.541322,0.216001
2,SVD++,0.682193,40.526490
